In [11]:
import torch, torch.nn as nn, torch.optim as optim

text  = 'hello world, this is a simple text generation using LSTMs.'
chars = sorted(set(text))
c2i = {c:i for i,c in enumerate(chars)}
i2c = {i:c for i,c in enumerate(chars)}

X,y=[],[]
for i in range(len(text)-10):
    X.append([c2i[c] for c in text[i:i+10]])
    y.append(c2i[text[i+10]])
X_t, y_t = torch.tensor(X), torch.tensor(y)

In [12]:
class LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.e = nn.Embedding(len(chars),16)
        self.l = nn.LSTM(16,128,batch_first=True)
        self.f = nn.Linear(128,len(chars))
    def forward(self,x):
        _,(h,_) = self.l(self.e(x))
        return self.f(h[-1])
    
model = LSTM()
opt = optim.Adam(model.parameters(),lr=0.01)
crit = nn.CrossEntropyLoss()

for e in range(50):
    opt.zero_grad()
    l = crit(model(X_t),y_t)
    l.backward()
    opt.step()
    if (e+1)%10==0:
        print(f'Loss: {l.item():.2f}')

Loss: 1.27
Loss: 0.10
Loss: 0.01
Loss: 0.00
Loss: 0.00


In [19]:
def generate(seed,n=50):
    s = [c2i[c] for c in seed]
    with torch.no_grad():
        for _ in range(n):
            p = model(torch.tensor([s[-10:]])).argmax(1).item()
            seed += i2c[p]
            s.append(p)
    return seed

print(generate('hello world',50))

hello world, this is a simple text generation using LSTMs...e
